# CodeBERT experiments on Kaggle

Before running, in the right-hand **Session options** panel:
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On (needed to clone the public repo and download `microsoft/codebert-base`)

Run the setup and GPU-check cells first. Then **Save Version -> Save & Run All (Commit)**.
When it finishes, download `results.zip` from the **Output** tab and unzip it into `experiments/`
in the repo. Each archive contains a new `kaggle_<timestamp>/` folder, preserving existing Mac results.

Runs CodeBERT LoRA and TF-IDF + LinearSVC (standard and balanced) on `modules_newlines.csv`: the combined module/newline treatment, with comments retained.
Uses the same rows and module names as the existing v2 module variant; whitespace-label exemptions remain unchanged. Qwen requires a separate CUDA port.


In [1]:
import shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/alexsmyr0/thesis-experiments"
REPO = Path("/tmp/thesis-experiments")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)], check=True)
# Use the notebook's interpreter and install every pipeline dependency.
# Stop immediately if setup fails. Existing compatible packages (including CUDA torch) stay installed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
# Kaggle may preinstall an old optional quantization backend that breaks PEFT adapter creation.
# CodeBERT uses unquantized weights, so remove that unused backend in this session.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", COMMIT)


Cloning into '/tmp/thesis-experiments'...


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Source commit: 8c17f1880196b8d4573d4109587d247685885f3b


In [2]:
# Check in a fresh process, matching the interpreter used by the training scripts.
# This avoids stale imports if Kaggle loaded a package before pip updated it.
import json

probe = subprocess.check_output([sys.executable, "-c", """
import importlib.metadata as metadata
import json
import torch
import transformers, peft, datasets, accelerate, sklearn, pandas, matplotlib, seaborn
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in Session options, then restart"
from transformers import TrainingArguments
assert TrainingArguments(output_dir='/tmp/kaggle-preflight', warmup_steps=0.1, report_to='none').get_warmup_steps(100) == 10
# Exercise adapter creation and a GPU backward pass before starting expensive jobs.
from transformers import RobertaConfig, RobertaForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model
model = get_peft_model(RobertaForSequenceClassification(RobertaConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=1,
    num_attention_heads=4, intermediate_size=64, num_labels=5)),
    LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
               lora_dropout=0.05, target_modules=['query', 'value'], bias='none')).cuda()
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
loss = model(input_ids=torch.full((4, 16), 2, device='cuda'),
             attention_mask=torch.ones((4, 16), dtype=torch.long, device='cuda'),
             labels=torch.tensor([0, 1, 2, 3], device='cuda')).loss.mean()
assert torch.isfinite(loss), 'LoRA GPU smoke test produced a non-finite loss'
loss.backward()
torch.cuda.synchronize()
del model, loss
torch.cuda.empty_cache()
packages = ['torch', 'transformers', 'peft', 'datasets', 'accelerate',
            'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']
print(json.dumps({
    'versions': {name: metadata.version(name) for name in packages},
    'cuda': torch.version.cuda,
    'lora_gpu_smoke_test': 'passed',
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}))
"""], text=True)
RUNTIME = json.loads(probe)
print(json.dumps(RUNTIME, indent=2))


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "peft": "0.21.2",
    "datasets": "4.8.5",
    "accelerate": "1.15.0",
    "scikit-learn": "1.6.1",
    "pandas": "2.3.3",
    "numpy": "2.1.3",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2"
  },
  "cuda": "13.0",
  "lora_gpu_smoke_test": "passed",
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ]
}


In [3]:
import os
from datetime import datetime, timezone

# Match Version 2 hardware/settings; run only the new combined treatment.
DATASETS = ["modules_newlines"]
CODEBERT = "lora"      # "full", "lora" or "both"
TFIDF = "svm"          # LinearSVC, standard and balanced class weights
FP16 = False           # Opt-in: faster on T4, but can shift scores

RUN_NAME = datetime.now(timezone.utc).strftime("kaggle_%Y%m%dT%H%M%S_%fZ")
OUT = Path("/kaggle/working/experiments") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "run_metadata.json").write_text(json.dumps({
    'run': RUN_NAME, 'repo': REPO_URL, 'commit': COMMIT,
    'datasets': DATASETS, 'codebert': CODEBERT, 'tfidf': TFIDF, 'fp16': FP16, **RUNTIME,
}, indent=2))
(OUT / "pip_freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))

for name in DATASETS:
    print(f"\n######## {name} ########", flush=True)
    env = {**os.environ,
           "THESIS_CSV": str(REPO / "data" / "v2" / f"{name}.csv"),
           "THESIS_OUT": str(OUT / name),
           "PY": sys.executable,
           "CODEBERT_ARGS": "--fp16" if FP16 else ""}
    subprocess.run(["sh", str(REPO / "scripts" / "run_all.sh"), f"--codebert={CODEBERT}", f"--tfidf={TFIDF}"], env=env, check=True)
    shutil.rmtree(OUT / name / "checkpoints", ignore_errors=True)  # keep the Output small



######## modules_newlines ########
== prepare_data.py
== tfidf_baselines.py --classifier svm
== codebert_finetune.py --mode lora
Run                      Thesis acc  Ours acc       Δ  Thesis F1  Ours F1       Δ
---------------------------------------------------------------------------------
tfidf_logreg                  0.950   not run
tfidf_logreg_balanced         0.940   not run
tfidf_svm                     0.972     0.885  -0.087      0.965    0.823  -0.142
tfidf_svm_balanced            0.972     0.887  -0.085      0.965    0.820  -0.145
tfidf_rf                      0.952   not run
tfidf_rf_balanced             0.948   not run
codebert_full                 0.988   not run
codebert_lora                 0.990     0.953  -0.037      0.993    0.921  -0.072
prompting_zeroshot            0.499   not run
prompting_fewshot_k1          0.591   not run
qwen_lora                     0.972   not run


In [4]:
for name in DATASETS:
    print(f"\n=== {name} ===")
    print((OUT / name / "logs" / "summary.txt").read_text())
shutil.make_archive("/kaggle/working/results", "zip", OUT.parent, OUT.name)
print(f"Wrote /kaggle/working/results.zip (contains {RUN_NAME}/)")



=== modules_newlines ===
Run                      Thesis acc  Ours acc       Δ  Thesis F1  Ours F1       Δ
---------------------------------------------------------------------------------
tfidf_logreg                  0.950   not run
tfidf_logreg_balanced         0.940   not run
tfidf_svm                     0.972     0.885  -0.087      0.965    0.823  -0.142
tfidf_svm_balanced            0.972     0.887  -0.085      0.965    0.820  -0.145
tfidf_rf                      0.952   not run
tfidf_rf_balanced             0.948   not run
codebert_full                 0.988   not run
codebert_lora                 0.990     0.953  -0.037      0.993    0.921  -0.072
prompting_zeroshot            0.499   not run
prompting_fewshot_k1          0.591   not run
qwen_lora                     0.972   not run

Wrote /kaggle/working/results.zip (contains kaggle_20261106T144239_908239Z/)
